# L5: Automate Event Planning

In this lesson, you will learn more about Tasks.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Crew, Task

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key,get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [6]:
 # Agent 2: Logistics Manager
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipmen"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [7]:
# Agent 3: Marketing and Communications Agent
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

## Creating Venue Pydantic Object

- Create a class `VenueDetails` using [Pydantic BaseModel](https://docs.pydantic.dev/latest/api/base_model/).
- Agents will populate this object with information about different venues by creating different instances of it.

In [8]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Creating Tasks
- By using `output_json`, you can specify the structure of the output you want.
- By using `output_file`, you can get your output in a file.
- By setting `human_input=True`, the task will ask for human feedback (whether you like the results or not) before finalising it.

In [9]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="venue_details.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

- By setting `async_execution=True`, it means the task can run in parallel with the tasks which come after it.

In [10]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    async_execution=True,
    agent=logistics_manager
)

In [11]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file="marketing_report.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

## Creating the Crew

**Note**: Since you set `async_execution=True` for `logistics_task` and `marketing_task` tasks, now the order for them does not matter in the `tasks` list.

In [12]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task, 
           logistics_task, 
           marketing_task],
    
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [13]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}

**Note 1**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Note 2**: 
- Since you set `human_input=True` for some tasks, the execution will ask for your input before it finishes running.
- When it asks for feedback, use your mouse pointer to first click in the text box before typing anything.

In [14]:
result = event_management_crew.kickoff(inputs=event_details)

 [DEBUG]: == Working Agent: Venue Coordinator
 [INFO]: == Starting Task: Find a venue in San Francisco that meets criteria for Tech Innovation Conference.


> Entering new CrewAgentExecutor chain...
I need to find a venue in San Francisco that meets the criteria for the Tech Innovation Conference.

Action: Search the internet
Action Input: {"search_query": "Tech Innovation Conference venue San Francisco"} 


Search results: Title: Book Modern Tech Innovation Hub & Event Space in San Francisco
Link: https://www.eventsinminutes.com/package/modern-tech-innovation-hub-event-space
Snippet: This modern tech innovation hub and event space is located in the Presidio district of San Francisco, California. It is a versatile venue specifically...
---
Title: TECHSPO San Francisco Technology ExpoTECHSPO San ...
Link: https://techsposanfrancisco.com/
Snippet: The annual TECHSPO San Francisco 2026 Technology Expo returns June 16th to 17th, 2026 to the Grand Hyatt San Francisco Hotel in San Francisco.

This is the agent final answer: Venue: Modern Tech Innovation Hub & Event Space
Location: Presidio district of San Francisco, California
Capacity: Banquet/Seated - 16 guests, Classroom - 16 guests
Features: Natural Light, Wheelchair Accessible, Elevator, On-site Parking (Paid)
Amenities: High-speed Wi-Fi, Presentation and Sound Systems, Coffee, Tea, Light Snacks, Comfortable Seating, Modern Decor
Availability: Monday to Friday, 8:00 AM to 7:00 PM
Price: $160.00/hour
Cancellation Policy: Full refund up to 30 days before, 50% refund 15-30 days before, No refund within 15 days
Rules: Only technology companies or tech content allowed, No smoking, illegal drugs, weapons, offensive materials, animals, certain permits required for outdoor use, Security cameras in common areas
FAQs: Vendors vetted and insured, Transparent pricing, Private event, Customizable packages, Instant confirmation, Cancellation and rescheduling policy, Ability to host at office or chosen venue, Support for large corpor

Thought: I need to gather information on the availability of the venue and delegate tasks related to catering and equipment setup for the event.

Action: Search the internet
Action Input: {"search_query": "Modern Tech Innovation Hub & Event Space availability for September 15, 2024"}Thought: I need to gather information about the Tech Innovation Conference and engage potential attendees effectively to meet the goal of at least 500 attendees.

Action: Read website content
Action Input: {"website_url": "https://techinnovationsummit2025.splashthat.com/"} 


Search results: Title: The Innovation Hub - Facebook
Link: https://www.facebook.com/InnovHubZA/posts/we-warmly-welcome-the-presidency-to-the-innovation-hub-conferencing-venue-as-the/616368647708371/
Snippet: Flexible Space: From cozy gatherings to team meetings and larger trainings, we offer a variety of flexible spaces to accommodate events from 2 ...
---
Title: S.2296 - National Defense Authorization Act for Fiscal Year 2026 ...
Link

Thought: I need to gather information on the availability of the venue and delegate tasks related to catering and equipment setup for the event.

Action: Search the internet
Action Input: {"search_query": "Modern Tech Innovation Hub & Event Space San Francisco availability September 15, 2024"} 


Search results: Title: TechCrunch Disrupt 2024
Link: https://techcrunch.com/events/tc-disrupt-2024/
Snippet: TechCrunch Disrupt 2026, October 13-15. TechCrunch Disrupt 2026. San Francisco, CA October 13 – 15, 2026. Register Now · Exhibit Now. Save up to $400, or up to ...
---
Title: AAPM Meetings
Link: https://www.aapm.org/meetings/
Snippet: The AAPM Annual Meeting & Exhibition held in late July or early August draws about 4,000 participants to the world's largest program of scientific, educational ...
---
Title: Museums on Us Map & Location Finder - Bank of America
Link: https://about.bankofamerica.com/en/making-an-impact/museums-on-us-find-locations-map
Snippet: Bank of America's Museums on 

Action Input: {"search_query": "Tech Innovation Summit 2025 Boston details"} 


Search results: Title: Boston Globe 2025 Tech Innovation Summit - MFN
Link: https://massfoundersnetwork.org/event/boston-globe-2025-tech-innovation-summit/
Snippet: The summit's programming will consist of dynamic discussions, expert-led panels, pitch contests, and more that dive into the technological advancements being ...
---
Title: Boston Globe's Tech Innovation Summit 2025
Link: https://www.greenoughagency.com/news-insights/boston-globes-tech-innovation-summit-2025
Snippet: At this year's Boston Tech Innovation Summit, the central theme was clear. It was innovation at the crossroads. And this encompassed many elements.
---
Title: 2025 Boston Innovation Summit: Shaping the Future of Dual-Use ...
Link: https://www.securingourfuture.us/p/2025-innovation-summit-shaping-the
Snippet: Save the date! New North Ventures invites you to the third annual 2025 Boston Innovation Summit on 6 February 2025.
---
Title:

Thought: I need to find out the availability of the Modern Tech Innovation Hub & Event Space for September 15, 2024, for an event with 500 participants.

Action: Search the internet
Action Input: {"search_query": "Modern Tech Innovation Hub & Event Space availability September 15, 2024"} 

I tried reusing the same input, I must stop using this action input. I'll try something else instead.



Thought: I need to find out the availability of the Modern Tech Innovation Hub & Event Space for September 15, 2024, for an event with 500 participants.

Action: Search the internet
Action Input: {"search_query": "Modern Tech Innovation Hub & Event Space contact information"}Thought: I need to gather information about the Tech Innovation Conference and engage potential attendees effectively to meet the goal of at least 500 attendees.

Action: Read website content
Action Input: {"website_url": "https://www.greenoughagency.com/news-insights/boston-globes-tech-innovation-summit-2025"} 

I tried reusi

I need to search for the availability and booking details of the Modern Tech Innovation Hub & Event Space for September 15, 2024.

Action: Search the internet
Action Input: {"search_query": "Modern Tech Innovation Hub & Event Space availability September 15, 2024 booking details"} 


Search results: Title: The Innovation Hub - Facebook
Link: https://www.facebook.com/InnovHubZA/posts/we-warmly-welcome-the-presidency-to-the-innovation-hub-conferencing-venue-as-the/616368647708371/
Snippet: The Innovation Hub Conferencing Venue offers state-of-the- art facilities, an inspiring atmosphere, and exceptional service to make your event a ...
---
Title: ICC - International Code Council - ICC
Link: https://www.iccsafe.org/
Snippet: Our mission is to provide the information, tools, and resources that members rely on, building professionals turn to, and the public trusts.
---
Title: Wiley Online Library | Scientific research articles, journals, books ...
Link: https://onlinelibrary.wiley.com/
Snip

Exception in thread Thread-6 (_execute):
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/local/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/site-packages/crewai/task.py", line 182, in _execute
    result = agent.execute_task(
             ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/crewai/agent.py", line 221, in execute_task
    result = self.agent_executor.invoke(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/langchain/chains/base.py", line 163, in invoke
    raise e
  File "/usr/local/lib/python3.11/site-packages/langchain/chains/base.py", line 153, in invoke
    self._call(inputs, run_manager=run_manager)
  File "/us

Final Answer: I was unable to find specific availability and booking details for the Modern Tech Innovation Hub & Event Space for September 15, 2024. I recommend reaching out directly to the venue for more information.

> Finished chain.
 

I was unable to find specific availability and booking details for the Modern Tech Innovation Hub & Event Space for September 15, 2024. I recommend reaching out directly to the venue for more information.

Thought: I need to gather information on the availability of the venue and delegate tasks related to catering and equipment setup for the event.

Action: Ask question to co-worker
Action Input: {"coworker": "Venue Coordinator", "question": "Is the Modern Tech Innovation Hub & Event Space available for booking on September 15, 2024?", "context": "I am the Logistics Manager and need to confirm the availability of the venue for an event with 500 participants."} 

After searching the internet, I was unable to find specific availability information for

Exception in thread Thread-5 (_execute):
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/local/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/site-packages/crewai/task.py", line 182, in _execute
    result = agent.execute_task(
             ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/crewai/agent.py", line 221, in execute_task
    result = self.agent_executor.invoke(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/langchain/chains/base.py", line 163, in invoke
    raise e
  File "/usr/local/lib/python3.11/site-packages/langchain/chains/base.py", line 153, in invoke
    self._call(inputs, run_manager=run_manager)
  File "/us

- Display the generated `venue_details.json` file.

In [15]:
import json
from pprint import pprint

with open('venue_details.json') as f:
   data = json.load(f)

pprint(data)

{'address': 'Presidio district of San Francisco, California',
 'booking_status': 'Available',
 'capacity': 16,
 'name': 'Modern Tech Innovation Hub & Event Space'}


- Display the generated `marketing_report.md` file.

**Note**: After `kickoff` execution has successfully ran, wait an extra 45 seconds for the `marketing_report.md` file to be generated. If you try to run the code below before the file has been generated, your output would look like:

```
marketing_report.md
```

If you see this output, wait some more and than try again.

In [16]:
from IPython.display import Markdown
Markdown("marketing_report.md")

marketing_report.md